# Colab SetupRun this notebook top to bottom every session.- Cells 1-5: mount Drive, clone/pull the repo, install dependencies, load API keys + git identity, confirm GPU.- Cell 6: restores processed data from a Drive backup if one exists (avoids a ~20-30 min reprocessing run).- Cells 8-10: **only run these if Cell 6 printed "No Drive backup found"** -- downloads from Kaggle, preprocesses, and backs up to Drive.- Cell 12: EDA (everyone runs this).- Cells 14-15: optional smoke tests confirming both models (BART, prompted LLM) work end-to-end. Not required every session.

In [ ]:
# 1. Mount Google Drive (persists checkpoints/outputs across sessions)from google.colab import drivedrive.mount('/content/drive')

In [ ]:
# 2. Clone the repo (or pull latest if already cloned this session)import osREPO_DIR = '/content/Review-Summarization-LLM'if os.path.exists(REPO_DIR):    %cd {REPO_DIR}    !git pull origin mainelse:    %cd /content    !git clone https://github.com/jrsheffie/Review-Summarization-LLM.git    %cd {REPO_DIR}

In [ ]:
# 3. Install dependencies!pip install -q -r requirements.txt

In [ ]:
# 4. Load API keys from Colab Secrets (key icon in left sidebar) and set git identityimport osfrom google.colab import userdataos.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')github_token = userdata.get('GITHUB_TOKEN')!git config --global user.email "jrsheffie@gmail.com"!git config --global user.name "jrsheffie"print('API keys and git identity loaded.')

In [ ]:
# 5. Confirm GPU runtime is enabled (Runtime -> Change runtime type -> T4 GPU)import torchprint('CUDA available:', torch.cuda.is_available())print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')

In [ ]:
# 6. Try restoring raw data + processed data from Drive backup first.# If no backup exists yet, this just prints a message and does nothing --# proceed to cells 8-10 to download and process from scratch.import osBACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'restored = Falseif os.path.exists(f'{BACKUP_DIR}/clean_reviews.csv'):    print('Found backup -- restoring processed data...')    !mkdir -p data/processed data/raw    !cp {BACKUP_DIR}/clean_reviews.csv {BACKUP_DIR}/train.csv {BACKUP_DIR}/val.csv {BACKUP_DIR}/test.csv {BACKUP_DIR}/product_batches.json data/processed/ 2>/dev/null    !cp {BACKUP_DIR}/Reviews.csv data/raw/ 2>/dev/null    restored = True    print('Restored from Drive backup.')else:    print('No Drive backup found yet -- run the download + preprocessing cells below.')

## Only run cells 8-10 if cell 6 printed "No Drive backup found" -- otherwise skip to cell 12

In [ ]:
# 7. Set up Kaggle API and download dataset (skip if restored above)from getpass import getpassimport ostoken = getpass("Paste your Kaggle API token: ").strip()os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:    f.write(token)!python data/download_dataset.py

In [ ]:
# 8. Run preprocessing (skip if restored above -- this takes ~20-30 min on the full dataset)!python data/preprocess.py --input data/raw/Reviews.csv --output-dir data/processed

In [ ]:
# 9. IMPORTANT: back this up to Drive immediately so you never redo steps 7-8 again!mkdir -p {BACKUP_DIR}!cp data/processed/*.csv data/processed/*.json {BACKUP_DIR}/!cp data/raw/Reviews.csv {BACKUP_DIR}/print('Backed up to Drive.')

## Everyone runs from here down

In [ ]:
# 10. Exploratory data analysis on the real cleaned data!python data/exploratory_analysis.py --input data/processed/clean_reviews.csv --output-dir outputs/figures

## Optional: smoke tests confirming both models work end-to-end (not required every session)

In [ ]:
# 11. BART baseline smoke test -- pretrained model, no fine-tuning yet.# Confirms the model implementation works end-to-end.from transformers import BartForConditionalGeneration, BartTokenizerimport pandas as pdtokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')df = pd.read_csv('data/processed/clean_reviews.csv')sample_reviews = df['Text'].sample(3, random_state=1).tolist()for i, review in enumerate(sample_reviews):    inputs = tokenizer(review, return_tensors='pt', max_length=1024, truncation=True)    summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=60, early_stopping=True)    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)    print(f'--- Review {i+1} ---')    print('Original:', review[:200], '...')    print('BART summary:', summary)    print()

In [ ]:
# 12. Prompted LLM smoke test -- uses the ANTHROPIC_API_KEY loaded in cell 4.import sys, jsonsys.path.insert(0, '.')from models.llm_prompting import summarize_productsbatches = json.load(open('data/processed/product_batches.json'))results = summarize_products(batches[:3])  # just 3 products as a smoke testfor r in results:    print(r['product_id'])    print(r.get('summary') or r.get('error'))    print('---')